# Laboratorio: Diseñando Combos para "Café COVA" ☕

**Contexto:**
La gerencia de la cafetería de la facultad quiere aumentar sus ingresos. Saben que los estudiantes compran cosas en grupo, pero no saben exactamente qué. Tu trabajo como Ingeniero de Datos es leer los registros crudos de la caja registradora, transformar los datos a una matriz transaccional y aplicar el algoritmo Apriori para encontrar los "Combos" más rentables.

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# FASE 1: EXTRACCIÓN (Datos de producción)
# El sistema punto de venta (POS) exporta los tickets como una lista de listas.
tickets_cafeteria = [
    ['Cafe Americano', 'Croissant', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Agua'],
    ['Cafe Americano', 'Croissant', 'Jugo de Naranja'],
    ['Te Verde', 'Galletas', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Galletas'],
    ['Cafe Americano', 'Croissant', 'Agua', 'Galletas'],
    ['Frappe Mocha', 'Brownie', 'Agua'],
    ['Te Verde', 'Ensalada', 'Agua'],
    ['Cafe Americano', 'Brownie', 'Jugo de Naranja'],
    ['Frappe Mocha', 'Muffin', 'Brownie']
]

### Reto 1: Ingeniería de Características (De Lista a Matriz)
El algoritmo Apriori no entiende listas de texto. Necesitas convertir `tickets_cafeteria` en un DataFrame de Pandas donde cada columna sea un producto y cada fila (ticket) tenga un `True` o `False` (One-Hot Encoding transaccional).

In [2]:
# TU CÓDIGO AQUÍ:
# Utiliza TransactionEncoder para crear df_cafe_bool
te = TransactionEncoder()
matriz_bool = te.fit_transform(tickets_cafeteria)
df_cafe_bool = pd.DataFrame(matriz_bool, columns=te.columns_)
display(df_cafe_bool)

,Agua,Brownie,Cafe Americano,Croissant,Ensalada,Frappe Mocha,Galletas,Jugo de Naranja,Muffin,Te Verde
0,True,False,True,True,False,False,False,False,False,False
1,True,False,False,False,False,True,False,False,True,False
2,False,False,True,True,False,False,False,True,False,False
3,True,False,False,False,False,False,True,False,False,True
4,False,False,False,False,False,True,True,False,True,False
5,True,False,True,True,False,False,True,False,False,False
6,True,True,False,False,False,True,False,False,False,False
7,True,False,False,False,True,False,False,False,False,True
8,False,True,True,False,False,False,False,True,False,False
9,False,True,False,False,False,True,False,False,True,False


### Reto 2: Extracción de Patrones Frecuentes
La gerencia te pide descartar las coincidencias raras. Extrae únicamente los `itemsets` (combinaciones de productos) que aparezcan en **al menos el 25% (0.25)** de los tickets.

In [3]:
# TU CÓDIGO AQUÍ:
# Usa la función apriori() pasándole tu df_cafe_bool. Guarda el resultado en itemsets_cafe
itemsets_cafe = apriori(df_cafe_bool, min_support=0.25, use_colnames=True)
display(itemsets_cafe)

,support,itemsets
0,0.6,(Agua)
1,0.4,(Cafe Americano)
2,0.4,(Frappe Mocha)


### Reto 3: Creación de Reglas de Asociación
A partir de los itemsets frecuentes, genera las reglas de asociación. Filtra el resultado para mostrar **únicamente** aquellas reglas que tengan una `Confianza` superior al 60% (0.6).

In [5]:
# TU CÓDIGO AQUÍ:
# Usa association_rules() y guarda el resultado en reglas_cafe
# Filtra las columnas para imprimir solo: antecedents, consequents, support, confidence, lift
reglas_cafe = association_rules(itemsets_cafe, metric="confidence", min_threshold=0.6)
display(reglas_cafe[["antecedents", "consequents", "support", "confidence", "lift"]])

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


### 🧠 Reto 4: Defensa de Ingeniería (Análisis de Negocio)

Revisa la tabla resultante de tus reglas y responde las siguientes preguntas para el gerente de la cafetería:

1. **La regla de mayor Confianza:** Identifica la regla que tiene el mayor nivel de Confianza. Traduce esa regla al español simple (ej. "Si un alumno compra X, existe un Y% de probabilidad de que también compre Z").
2. **El engaño del Soporte vs Lift:** ¿Qué producto de la cafetería se vende muchísimo (alto soporte), pero no deberíamos ponerlo en promoción (combo) con el Frappe Mocha porque su Lift es bajo o cercano a 1? Explica matemáticamente por qué.

**1.** Hay dos reglas empatadas con 100% de confianza: Croissant → Cafe Americano y Muffin → Frappe Mocha. O sea, si un alumno compra un Croissant, hay un 100% de probabilidad de que también compre un Cafe Americano (y lo mismo con el Muffin y el Frappe Mocha).

**2.** El Agua. Es lo que más se vende (soporte de 0.6, sale en 6 de 10 tickets), pero con el Frappe Mocha solo aparece junta en 2 tickets. La confianza de Frappe Mocha → Agua es 0.2 / 0.4 = 0.5, y el lift es 0.5 / 0.6 = 0.83, menor a 1. Eso significa que comprar un Frappe no hace más probable que compren Agua, al contrario, es un poco menos probable que con cualquier cliente.